# PrimeKG + ECoGraph colorectal cancer subtypes

Run this notebook beside the `kg_data` folder used by Untitled.ipynb, or edit the paths in the first cell. The saved notebook showed four ECoGraph labels and a blank sample–gene export. This merge derives mutation observations from the binary COAD matrix in every run. It inventories the optional Neptune edge files separately, so a missing, empty, or partial export cannot change the mutation denominator.

Each subtype is a distinct disease node with an ECoGraph identifier. Each mutation edge means at least one matched TCGA-COAD patient in that subtype had a positive call for the gene. The count and fraction describe this cohort. They do not measure subtype enrichment, causal effects, or drug indications.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
from urllib.request import urlretrieve

import pandas as pd

data_dir = Path("kg_data")
primekg_path = data_dir / "kg.csv"
ecograph_dir = data_dir / "ecograph"
out_dir = data_dir / "primekg_ecograph"
eco_url = "https://raw.githubusercontent.com/collaborativebioinformatics/ECoGraph/main/data/"
hgnc_url = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"

# Optional. After inspecting crc_candidates below, set this to the reviewed
# PrimeKG node_index for the broad CRC/COAD parent and rerun from this cell.
crc_parent_index = None

if not primekg_path.is_file():
    raise FileNotFoundError(f"Point primekg_path at your downloaded PrimeKG kg.csv: {primekg_path}")

def read_required(path, url, columns=None, sep=","):
    if not path.exists() or path.stat().st_size == 0:
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + ".part")
        try:
            urlretrieve(url, temporary)
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)
    try:
        frame = pd.read_csv(path, dtype="string", keep_default_na=False, sep=sep, usecols=columns)
    except pd.errors.EmptyDataError as error:
        raise ValueError(f"No readable CSV data in {path}. Replace it with the file at {url}") from error
    if frame.empty:
        raise ValueError(f"No data rows in {path}. Replace it with the file at {url}")
    return frame

gene_parts = sorted(ecograph_dir.glob("edges_has_gene_part*.csv"))
expected_gene_part = ecograph_dir / "edges_has_gene_part001.csv"
for path in sorted(set(gene_parts + [expected_gene_part] + [
    ecograph_dir / "edges_in_pathway.csv", ecograph_dir / "edges_is_of_age.csv"
])):
    if not path.is_file():
        status = "missing"
    else:
        try:
            preview = pd.read_csv(path, nrows=1, dtype="string")
            status = "header only" if preview.empty else "has rows"
        except pd.errors.EmptyDataError:
            status = "empty or blank"
        except (pd.errors.ParserError, UnicodeError):
            status = "unreadable"
    print(f"{path.name}: {status} (outside this merge)")

edges_has_gene_part001.csv: empty or blank (outside this merge)
edges_in_pathway.csv: has rows (outside this merge)
edges_is_of_age.csv: has rows (outside this merge)


In [2]:
# Collect PrimeKG nodes from both ends of the published edge list.
node_fields = ["index", "id", "type", "name", "source"]
endpoint_fields = [f"{side}_{field}" for side in ("x", "y") for field in node_fields]
nodes = pd.DataFrame(columns=node_fields)
for chunk in pd.read_csv(primekg_path, usecols=endpoint_fields, chunksize=250_000,
                         dtype="string", keep_default_na=False):
    endpoints = [
        chunk[[f"{side}_{field}" for field in node_fields]].set_axis(node_fields, axis=1)
        for side in ("x", "y")
    ]
    nodes = pd.concat([nodes, *endpoints], ignore_index=True).drop_duplicates(ignore_index=True)

if nodes.empty:
    raise ValueError("PrimeKG has no nodes.")
nodes["index"] = nodes["index"].astype("int64")
assert nodes["index"].is_unique, "Conflicting metadata for a PrimeKG node index"
nodes = nodes.rename(columns={field: f"node_{field}" for field in node_fields})
nodes["graph_source"] = "PrimeKG"
crc_candidates = nodes.loc[
    nodes["node_type"].eq("disease") &
    nodes["node_name"].str.contains(r"colorectal|colon (?:cancer|carcinoma|adenocarcinoma)", case=False)
]
display(crc_candidates[["node_index", "node_id", "node_name", "node_source"]])

,node_index,node_id,node_name,node_source
27258,28849,5575,colorectal cancer,MONDO
34073,27819,18604_23113,familial colorectal cancer,MONDO_grouped
35137,28991,14038_12820_12953_12132_12673_12821_12822_1282...,"colorectal cancer, susceptibility to",MONDO_grouped
35332,29203,9521,"leukemia, acute myelocytic, with polyposis col...",MONDO
36351,30299,12249_13699_13725_13695_13710_13196,"colorectal cancer, hereditary nonpolyposis,",MONDO_grouped
50564,35710,5484,colorectal adenoma,MONDO
50686,35880,2271,colon adenocarcinoma,MONDO
50687,35881,20794,colorectal medullary carcinoma,MONDO
50784,36022,3885,colorectal lipoma,MONDO
50786,36024,3299,colorectal leiomyoma,MONDO


In [3]:
# Join subtype assignments to mutation profiles at the TCGA patient level.
subtype_path = ecograph_dir / "vertices_subtype.csv"
assignment_path = ecograph_dir / "edges_has_subtype.csv"
mutation_path = ecograph_dir / "coad_mut_data.csv"
subtypes = read_required(subtype_path, eco_url + "neptune_csv/" + subtype_path.name, ["~id", "~label"])
subtype_ids = sorted(subtypes["~id"])
assert subtypes["~id"].is_unique and subtypes["~label"].eq("Subtype").all()
assert all(value.startswith("subtype:") for value in subtype_ids)

assignments = read_required(assignment_path, eco_url + "neptune_csv/" + assignment_path.name,
                            ["~from", "~to", "~label"])
assert assignments["~label"].eq("HAS_SUBTYPE").all()
assignments = assignments.rename(columns={"~from": "patient_id", "~to": "subtype"})
assignments["patient_id"] = assignments["patient_id"].str.removeprefix("sample:")
assignments = assignments[["patient_id", "subtype"]].drop_duplicates()
assert assignments["patient_id"].str.fullmatch(r"TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}").all()
assert assignments["patient_id"].is_unique, "A patient has conflicting subtype assignments"
assert set(assignments["subtype"]) <= set(subtype_ids)

mutations = read_required(mutation_path, eco_url + mutation_path.name)
with mutation_path.open(newline="", encoding="utf-8-sig") as stream:
    header = next(csv.reader(stream))
assert len(header) == len(set(header)), "Duplicate gene columns in mutation matrix"
barcode = mutations.pop("Tumor_Sample_Barcode")
patient_id = mutations.pop("patient_id")
assert patient_id.str.fullmatch(r"TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}").all()
assert barcode.str.fullmatch(r"TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-01[A-Z0-9](?:-[A-Z0-9]+)*").all()
assert barcode.str.slice(0, 12).eq(patient_id).all()
assert patient_id.is_unique, "Review multiple tumor profiles per patient before aggregating"
assert mutations.shape[1] and mutations.isin(["0", "1"]).all().all(), "Expected binary gene calls"
mutations = mutations.astype("uint8").set_axis(patient_id, axis=0)

labeled = assignments.set_index("patient_id").join(
    pd.Series(True, index=mutations.index, name="profile_present"), how="left", validate="1:1"
)
cohort = labeled[labeled["profile_present"].eq(True)]
coverage = pd.DataFrame({
    "n_labeled": labeled.groupby("subtype").size(),
    "n_profiled": cohort.groupby("subtype").size(),
}).reindex(subtype_ids).fillna(0).astype(int)
display(coverage)
print("Mutation profiles without a subtype:", len(mutations.index.difference(labeled.index)))
assert not cohort.empty, "No patient overlap between the mutation matrix and subtype assignments"

,n_labeled,n_profiled
subtype,,
subtype:GI.CIN,224,224
subtype:GI.GS,49,49
subtype:GI.HM-SNV,6,6
subtype:GI.HM-indel,60,60


Mutation profiles without a subtype: 65


In [4]:
# Map exact HGNC-approved symbols to NCBI Gene IDs, then to unique PrimeKG nodes.
# The COAD matrix has gene symbols and the original ECoGraph vertices have no stable gene ID.
hgnc_path = ecograph_dir / "hgnc_complete_set.txt"
hgnc = read_required(hgnc_path, hgnc_url, ["symbol", "entrez_id", "status", "hgnc_id"], sep="	")
approved = hgnc.loc[hgnc["status"].eq("Approved"), ["symbol", "entrez_id", "hgnc_id"]].drop_duplicates()
approved = approved.loc[~approved["symbol"].duplicated(keep=False)]
prime_genes = nodes.loc[
    nodes["node_type"].eq("gene/protein") & nodes["node_source"].eq("NCBI"),
    ["node_id", "node_index"]
]
prime_genes = prime_genes.loc[~prime_genes["node_id"].duplicated(keep=False)]
gene_map = pd.DataFrame({"symbol": mutations.columns}).merge(
    approved, on="symbol", how="left", validate="1:1"
)
gene_map = gene_map.merge(
    prime_genes, left_on="entrez_id", right_on="node_id", how="left", validate="m:1"
)
gene_map["node_index"] = gene_map["node_index"].astype("Int64")
gene_map["mapping_status"] = "no_unique_approved_symbol"
gene_map.loc[gene_map["hgnc_id"].notna(), "mapping_status"] = "no_unique_primekg_ncbi_target"
gene_map.loc[gene_map["node_index"].notna(), "mapping_status"] = "mapped"
mapped = gene_map.loc[gene_map["mapping_status"].eq("mapped")].copy()
duplicate_target = mapped["node_index"].duplicated(keep=False)
gene_map.loc[mapped.index[duplicate_target], "mapping_status"] = "multiple_matrix_symbols_for_target"
mapped = mapped.loc[~duplicate_target].set_index("symbol")
display(gene_map["mapping_status"].value_counts())
assert not mapped.empty, "No usable gene mappings"

mapping_status
mapped                           18575
no_unique_approved_symbol         2538
no_unique_primekg_ncbi_target      219
Name: count, dtype: int64

In [5]:
# Count each patient once per gene and retain all positive observations.
counts = mutations.loc[cohort.index, mapped.index].groupby(cohort["subtype"]).sum()
evidence = counts.rename_axis(index="subtype", columns="symbol").stack().rename("n_mutated").reset_index()
evidence = evidence.loc[evidence["n_mutated"].gt(0)].merge(
    mapped[["node_index", "entrez_id", "hgnc_id"]], left_on="symbol", right_index=True, validate="m:1"
).rename(columns={"node_index": "y_index"})
evidence["n_profiled"] = evidence["subtype"].map(coverage["n_profiled"])
evidence["mutation_fraction"] = evidence["n_mutated"] / evidence["n_profiled"]
evidence["mapping_method"] = "HGNC approved symbol to NCBI Gene ID"
evidence["evidence_source"] = "ECoGraph TCGA-COAD"
evidence["evidence_files"] = f"{mutation_path.name} + {assignment_path.name}"

new_nodes = pd.DataFrame({
    "node_id": ["ECoGraph:" + label for label in subtype_ids],
    "node_index": range(int(nodes["node_index"].max()) + 1,
                        int(nodes["node_index"].max()) + 1 + len(subtype_ids)),
    "node_type": "disease",
    "node_name": ["Colorectal cancer molecular subtype " + label.split(":", 1)[1] for label in subtype_ids],
    "node_source": "ECoGraph",
    "graph_source": "ECoGraph",
    "ecograph_subtype": subtype_ids,
    "node_kind": "molecular_subtype",
})
subtype_index = new_nodes.set_index("ecograph_subtype")["node_index"]
evidence["x_index"] = evidence["subtype"].map(subtype_index)
evidence["relation"] = "ecograph_subtype_has_observed_mutation_in_gene"
assert not evidence.empty
assert evidence["n_mutated"].le(evidence["n_profiled"]).all()
assert evidence["mutation_fraction"].between(0, 1, inclusive="right").all()
assert not evidence.duplicated(["x_index", "relation", "y_index"]).any()
assert set(evidence["y_index"]) <= set(prime_genes["node_index"])
assert set(evidence["x_index"]) <= set(new_nodes["node_index"])
assert set(nodes["node_index"]).isdisjoint(new_nodes["node_index"])

edge_fields = ["x_index", "relation", "y_index"]
added_edges = evidence[edge_fields].copy()
if crc_parent_index is not None:
    if crc_parent_index not in set(crc_candidates["node_index"]):
        raise ValueError("Select a reviewed disease index from crc_candidates.")
    parent_edges = new_nodes[["node_index"]].rename(columns={"node_index": "x_index"})
    parent_edges["relation"] = "ecograph_molecular_subtype_of"
    parent_edges["y_index"] = crc_parent_index
    added_edges = pd.concat([added_edges, parent_edges[edge_fields]], ignore_index=True)

display(evidence.groupby("subtype").size().reindex(subtype_ids, fill_value=0).rename("mutation_edges"))
display(evidence.sort_values(["subtype", "n_mutated"], ascending=[True, False])
        .groupby("subtype").head(5)[["subtype", "symbol", "n_mutated", "n_profiled", "mutation_fraction"]])

subtype
subtype:GI.CIN          9763
subtype:GI.GS           3741
subtype:GI.HM-SNV      11421
subtype:GI.HM-indel    15580
Name: mutation_edges, dtype: int64

,subtype,symbol,n_mutated,n_profiled,mutation_fraction
1117,subtype:GI.CIN,APC,185,224,0.825893
11,subtype:GI.CIN,TP53,170,224,0.758929
1995,subtype:GI.CIN,KRAS,97,224,0.433036
150,subtype:GI.CIN,TTN,94,224,0.419643
60,subtype:GI.CIN,PIK3CA,56,224,0.25
19692,subtype:GI.GS,APC,40,49,0.816327
20570,subtype:GI.GS,KRAS,35,49,0.714286
18725,subtype:GI.GS,TTN,31,49,0.632653
18635,subtype:GI.GS,PIK3CA,22,49,0.44898
21405,subtype:GI.GS,SOX9,15,49,0.306122


In [6]:
# Export a compact graph and evidence sidecars. Preserve PrimeKG's existing directions.
# Re-running overwrites these outputs. The source kg.csv is read-only here.
out_dir.mkdir(parents=True, exist_ok=True)
pd.concat([nodes, new_nodes], ignore_index=True).to_csv(out_dir / "nodes.csv", index=False)
edge_path = out_dir / "edges.csv"
temp_path = out_dir / "edges.csv.part"
with temp_path.open("w") as stream:
    export_fields = edge_fields + ["display_relation"]
    stream.write(",".join(export_fields + ["graph_source"]) + "\n")
    base_rows = 0
    for chunk in pd.read_csv(primekg_path, usecols=export_fields, dtype="string",
                             keep_default_na=False, chunksize=250_000):
        assert chunk["x_index"].astype("int64").isin(nodes["node_index"]).all()
        assert chunk["y_index"].astype("int64").isin(nodes["node_index"]).all()
        chunk[export_fields].assign(graph_source="PrimeKG").to_csv(stream, header=False, index=False)
        base_rows += len(chunk)
    added_edges.assign(display_relation=added_edges["relation"], graph_source="ECoGraph")[
        export_fields + ["graph_source"]
    ].to_csv(stream, header=False, index=False)
temp_path.replace(edge_path)

evidence.to_csv(out_dir / "subtype_gene_evidence.csv", index=False)
coverage.rename_axis("subtype").to_csv(out_dir / "cohort_coverage.csv")
gene_map.to_csv(out_dir / "gene_mapping_audit.csv", index=False)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

sources = {
    primekg_path: "https://dataverse.harvard.edu/api/access/datafile/6180620",
    mutation_path: eco_url + mutation_path.name,
    assignment_path: eco_url + "neptune_csv/" + assignment_path.name,
    subtype_path: eco_url + "neptune_csv/" + subtype_path.name,
    hgnc_path: hgnc_url,
}
manifest = {
    "inputs": [{"file": str(path), "reference_url": url, "sha256": sha256(path)}
               for path, url in sources.items()],
    "primekg_rows": base_rows,
    "added_rows": len(added_edges),
    "reviewed_crc_parent_index": crc_parent_index,
    "mutation_rule": "One primary-tumor binary mutation profile per subtype-labeled patient; retain positive counts.",
    "denominator": "Subtype-labeled patients with a mutation matrix row, including rows with no positive calls.",
    "gene_mapping": "Exact HGNC-approved symbol via NCBI Gene ID to an unambiguous PrimeKG node.",
}
(out_dir / "provenance.json").write_text(json.dumps(manifest, indent=2))
print(f"Saved {base_rows:,} PrimeKG rows and {len(added_edges):,} ECoGraph rows to {edge_path}")

Saved 8,100,498 PrimeKG rows and 40,505 ECoGraph rows to kg_data/primekg_ecograph/edges.csv


In [8]:
import pandas as pd

# Read the saved graph. Scan edges in chunks because PrimeKG is large.
saved_nodes = pd.read_csv(out_dir / "nodes.csv", low_memory=False)
assert saved_nodes["node_index"].is_unique, "Duplicate node indices"

node_ids = set(saved_nodes["node_index"])
new_edge_chunks = []

for chunk in pd.read_csv(
    out_dir / "edges.csv",
    usecols=["x_index", "relation", "y_index", "graph_source"],
    chunksize=250_000,
):
    assert chunk["x_index"].isin(node_ids).all(), "Missing source node"
    assert chunk["y_index"].isin(node_ids).all(), "Missing target node"
    new_edge_chunks.append(chunk.loc[chunk["graph_source"].eq("ECoGraph")])

new_edges = pd.concat(new_edge_chunks, ignore_index=True)
assert len(new_edges) == len(added_edges), "Exported edge count differs from the merge"
assert not new_edges.duplicated(["x_index", "relation", "y_index"]).any(), "Duplicate new edges"

mutation_relation = "ecograph_subtype_has_observed_mutation_in_gene"
mutation_edges = new_edges.loc[new_edges["relation"].eq(mutation_relation)]

subtype_nodes = saved_nodes.loc[saved_nodes["node_kind"].eq("molecular_subtype")]
assert len(subtype_nodes) == 4, "Expected four ECoGraph subtype nodes"
assert subtype_nodes["node_type"].eq("disease").all()
assert subtype_nodes["ecograph_subtype"].is_unique

subtype_by_index = subtype_nodes.set_index("node_index")["ecograph_subtype"]
node_type = saved_nodes.set_index("node_index")["node_type"]
assert mutation_edges["x_index"].isin(subtype_by_index.index).all()
assert mutation_edges["y_index"].map(node_type).eq("gene/protein").all()

evidence = pd.read_csv(out_dir / "subtype_gene_evidence.csv")
assert not evidence.duplicated(["x_index", "y_index"]).any()
assert evidence["x_index"].map(subtype_by_index).eq(evidence["subtype"]).all()
assert set(zip(mutation_edges["x_index"], mutation_edges["y_index"])) == set(
    zip(evidence["x_index"], evidence["y_index"])
), "Saved mutation edges and evidence disagree"

# Recount directly from the binary matrix for each subtype.
summary = []
for index, subtype in subtype_by_index.items():
    patients = cohort.index[cohort["subtype"].eq(subtype)]
    expected = mutations.loc[patients, mapped.index].sum(axis=0)
    expected = expected[expected.gt(0)].sort_index()

    rows = evidence.loc[evidence["x_index"].eq(index)]
    actual = rows.set_index("symbol")["n_mutated"].sort_index()
    assert len(expected) > 0, f"{subtype}: no mapped mutation edges"
    assert expected.to_dict() == actual.to_dict(), f"{subtype}: gene counts differ"
    assert rows["y_index"].eq(rows["symbol"].map(mapped["node_index"])).all()
    assert rows["n_profiled"].eq(len(patients)).all()
    fraction_error = (rows["mutation_fraction"] - rows["n_mutated"] / len(patients)).abs()
    assert fraction_error.lt(1e-12).all(), f"{subtype}: incorrect mutation fractions"

    summary.append({
        "subtype": subtype,
        "profiled_patients": len(patients),
        "mutation_edges": len(rows),
        "genes_seen_once": rows["n_mutated"].eq(1).sum(),
        "genes_seen_in_2plus_patients": rows["n_mutated"].ge(2).sum(),
    })

display(pd.DataFrame(summary).set_index("subtype"))
display(
    evidence.loc[evidence["symbol"].isin(["APC", "TP53"]),
                 ["subtype", "symbol", "n_mutated", "n_profiled", "mutation_fraction"]]
    .sort_values(["symbol", "subtype"])
)
print(f"Passed. Verified {len(mutation_edges):,} subtype–gene edges.")

,profiled_patients,mutation_edges,genes_seen_once,genes_seen_in_2plus_patients
subtype,,,,
subtype:GI.CIN,224,9763,4813,4950
subtype:GI.GS,49,3741,2876,865
subtype:GI.HM-SNV,6,11421,5027,6394
subtype:GI.HM-indel,60,15580,2883,12697


,subtype,symbol,n_mutated,n_profiled,mutation_fraction
822,subtype:GI.CIN,APC,185,224,0.825893
10170,subtype:GI.GS,APC,40,49,0.816327
14379,subtype:GI.HM-SNV,APC,6,6,1.000000
25996,subtype:GI.HM-indel,APC,23,60,0.383333
8,subtype:GI.CIN,TP53,170,224,0.758929
9770,subtype:GI.GS,TP53,7,49,0.142857
13514,subtype:GI.HM-SNV,TP53,2,6,0.333333
24935,subtype:GI.HM-indel,TP53,17,60,0.283333


Passed. Verified 40,505 subtype–gene edges.


A small denominator matters. One observation among six profiled patients contributes about 17%. Use the counts alongside fractions when comparing subtypes. For evaluation against held-out patients, split patients before aggregating the mutation edges.

The optional parent edge records a reviewed hierarchy link. Each subtype keeps its own index, and PrimeKG drug indication edges stay attached to their original endpoints.

References

- [ECoGraph and its TCGA-COAD inputs](https://github.com/collaborativebioinformatics/ECoGraph)
- [PrimeKG](https://github.com/mims-harvard/PrimeKG)
- [HGNC approved gene reference](https://www.genenames.org/download/statistics-and-files/)

In [9]:
nodes = pd.read_csv(out_dir / "nodes.csv", low_memory=False)
display(nodes.loc[
    nodes["node_source"].eq("ECoGraph") & nodes["node_type"].eq("disease"),
    ["node_index", "node_id", "node_name", "node_type"],
])

,node_index,node_id,node_name,node_type
129375,129375,ECoGraph:subtype:GI.CIN,Colorectal cancer molecular subtype GI.CIN,disease
129376,129376,ECoGraph:subtype:GI.GS,Colorectal cancer molecular subtype GI.GS,disease
129377,129377,ECoGraph:subtype:GI.HM-SNV,Colorectal cancer molecular subtype GI.HM-SNV,disease
129378,129378,ECoGraph:subtype:GI.HM-indel,Colorectal cancer molecular subtype GI.HM-indel,disease


In [10]:
display(
    new_edges.assign(subtype=new_edges["x_index"].map(subtype_by_index))
    .groupby(["subtype", "relation"])
    .size()
    .unstack(fill_value=0)
)

relation,ecograph_subtype_has_observed_mutation_in_gene
subtype,
subtype:GI.CIN,9763
subtype:GI.GS,3741
subtype:GI.HM-SNV,11421
subtype:GI.HM-indel,15580
